# 06 OpenOximetry PPG Alignment Pilot (Phase 2)

Audit timing alignment between PPG headers and 2Hz files, then freeze a conservative eligible encounter subset.

Inputs from: OpenOximetry waveform and bloodgas sources, informed by notebook 05 audit outputs.
Outputs to: `outputs/openox_alignment_audit.csv` and `outputs/openox_ppg_eligible_encounters.csv`.
Next: notebook 07 for sample-anchored reference pairing.

In [ ]:
from pathlib import Path
import csv
import datetime as dt

import pandas as pd

start = Path.cwd().resolve()
PROJECT_ROOT = None
for candidate in [start, *start.parents]:
    if (candidate / "data" / "physionet.org" / "files" / "openox-repo" / "1.1.1").is_dir() and (candidate / "notebooks").is_dir():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise RuntimeError("Could not find project root containing OpenOximetry data and notebooks/")

OPENOX_DIR = PROJECT_ROOT / "data" / "physionet.org" / "files" / "openox-repo" / "1.1.1"
WAVEFORMS_DIR = OPENOX_DIR / "waveforms"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

BLOODGAS_CSV = OPENOX_DIR / "bloodgas.csv"
OPENOX_ALIGNMENT_AUDIT_CSV = OUTPUTS_DIR / "openox_alignment_audit.csv"
OPENOX_PPG_ELIGIBLE_CSV = OUTPUTS_DIR / "openox_ppg_eligible_encounters.csv"

if not WAVEFORMS_DIR.is_dir():
    raise FileNotFoundError(f"Missing waveforms directory: {WAVEFORMS_DIR}")
if not BLOODGAS_CSV.is_file():
    raise FileNotFoundError(f"Missing bloodgas CSV: {BLOODGAS_CSV}")

print(f"Project root: {PROJECT_ROOT}")
print(f"Waveforms dir: {WAVEFORMS_DIR}")
print(f"Outputs dir: {OUTPUTS_DIR}")

## Parse Available Encounters And Timing Signals

Define time-parsing helpers, inspect available PPG and 2Hz files, and compute encounter-level timing anchors used for eligibility checks.

Expected result: encounter-to-file maps and timing parsers ready for audit construction.

In [ ]:
def parse_time_of_day_to_seconds(value: str | None) -> float | None:
    if value is None:
        return None
    text = str(value).strip()
    if text == "":
        return None
    for fmt in ("%H:%M:%S.%f", "%H:%M:%S"):
        try:
            parsed = dt.datetime.strptime(text, fmt)
            return parsed.hour * 3600.0 + parsed.minute * 60.0 + parsed.second + parsed.microsecond / 1_000_000.0
        except ValueError:
            continue
    return None


def parse_timestamp_to_seconds_of_day(value: str | None) -> tuple[str | None, float | None]:
    if value is None:
        return (None, None)
    text = str(value).strip()
    if text == "":
        return (None, None)

    parsed = pd.to_datetime([text], errors="coerce")
    ts = parsed[0]
    if pd.isna(ts):
        return (text, None)

    seconds = ts.hour * 3600.0 + ts.minute * 60.0 + ts.second + ts.microsecond / 1_000_000.0
    return (text, seconds)


def min_circular_day_delta_seconds(a: float | None, b: float | None) -> float | None:
    if a is None or b is None:
        return None
    raw = abs(a - b)
    return min(raw, 86400.0 - raw)


def parse_ppg_header_start_time(hea_path: Path) -> str | None:
    with hea_path.open("r", encoding="utf-8", errors="ignore") as f:
        first_line = f.readline().strip()

    if first_line == "":
        return None

    tokens = first_line.split()
    for token in tokens:
        if token.count(":") == 2:
            return token
    return None


def parse_2hz_timing(csv_path: Path) -> dict[str, object]:
    first_timestamp_text = None
    first_seconds = None
    last_timestamp_text = None
    last_seconds = None
    sample_non_null_count = 0
    row_count = 0

    with csv_path.open("r", encoding="utf-8", errors="ignore", newline="") as f:
        reader = csv.DictReader(f)
        if reader.fieldnames is None:
            return {
                "hz2_row_count": 0,
                "hz2_first_timestamp": None,
                "hz2_last_timestamp": None,
                "hz2_first_time_seconds": None,
                "hz2_last_time_seconds": None,
                "hz2_duration_seconds": None,
                "hz2_non_null_sample_count": 0,
            }

        has_sample = "Sample" in reader.fieldnames

        for row in reader:
            row_count += 1
            ts_text, ts_seconds = parse_timestamp_to_seconds_of_day(row.get("Timestamp"))

            if ts_text is not None:
                if first_timestamp_text is None:
                    first_timestamp_text = ts_text
                    first_seconds = ts_seconds
                last_timestamp_text = ts_text
                last_seconds = ts_seconds

            if has_sample:
                sample_val = row.get("Sample")
                if sample_val is not None and str(sample_val).strip() != "":
                    sample_non_null_count += 1

    duration_seconds = None
    if first_seconds is not None and last_seconds is not None:
        duration_seconds = (pd.to_datetime(last_timestamp_text) - pd.to_datetime(first_timestamp_text)).total_seconds()

    return {
        "hz2_row_count": int(row_count),
        "hz2_first_timestamp": first_timestamp_text,
        "hz2_last_timestamp": last_timestamp_text,
        "hz2_first_time_seconds": first_seconds,
        "hz2_last_time_seconds": last_seconds,
        "hz2_duration_seconds": duration_seconds,
        "hz2_non_null_sample_count": int(sample_non_null_count),
    }


ppg_hea_files = sorted(WAVEFORMS_DIR.rglob("*_ppg.hea"))
hz2_csv_files = sorted(WAVEFORMS_DIR.rglob("*_2hz.csv"))

ppg_hea_by_encounter = {p.name.replace("_ppg.hea", ""): p for p in ppg_hea_files}
hz2_by_encounter = {p.name.replace("_2hz.csv", ""): p for p in hz2_csv_files}

print(f"PPG headers found: {len(ppg_hea_by_encounter)}")
print(f"2Hz files found: {len(hz2_by_encounter)}")

## Build Alignment Audit And Eligible Subset

Combine PPG/2Hz availability, bloodgas So2 presence, and time-delta eligibility into an auditable table, then export the provisional PPG-eligible encounter set.

In [ ]:
bloodgas_df = pd.read_csv(BLOODGAS_CSV, low_memory=False)
bloodgas_df["so2_numeric"] = pd.to_numeric(bloodgas_df.get("so2"), errors="coerce")
bloodgas_encounters_with_so2 = set(
    bloodgas_df.loc[bloodgas_df["so2_numeric"].notna(), "encounter_id"].astype(str)
)

all_encounters = sorted(set(ppg_hea_by_encounter) | set(hz2_by_encounter) | bloodgas_encounters_with_so2)

timing_threshold_seconds = 10.0
audit_rows = []
for encounter_id in all_encounters:
    ppg_path = ppg_hea_by_encounter.get(encounter_id)
    hz2_path = hz2_by_encounter.get(encounter_id)

    ppg_start_time_text = None
    ppg_start_time_seconds = None
    if ppg_path is not None:
        ppg_start_time_text = parse_ppg_header_start_time(ppg_path)
        ppg_start_time_seconds = parse_time_of_day_to_seconds(ppg_start_time_text)

    hz2_info = {
        "hz2_row_count": 0,
        "hz2_first_timestamp": None,
        "hz2_last_timestamp": None,
        "hz2_first_time_seconds": None,
        "hz2_last_time_seconds": None,
        "hz2_duration_seconds": None,
        "hz2_non_null_sample_count": 0,
    }
    if hz2_path is not None:
        hz2_info = parse_2hz_timing(hz2_path)

    abs_delta_seconds = min_circular_day_delta_seconds(ppg_start_time_seconds, hz2_info["hz2_first_time_seconds"])

    has_ppg_hea = ppg_path is not None
    has_2hz_csv = hz2_path is not None
    has_bloodgas_so2 = encounter_id in bloodgas_encounters_with_so2
    timing_eligible = abs_delta_seconds is not None and abs_delta_seconds <= timing_threshold_seconds
    provisional_ppg_eligible = has_ppg_hea and has_2hz_csv and has_bloodgas_so2 and timing_eligible

    audit_rows.append(
        {
            "encounter_id": encounter_id,
            "has_ppg_hea": bool(has_ppg_hea),
            "has_2hz_csv": bool(has_2hz_csv),
            "has_bloodgas_so2": bool(has_bloodgas_so2),
            "ppg_hea_path": str(ppg_path) if ppg_path is not None else None,
            "hz2_csv_path": str(hz2_path) if hz2_path is not None else None,
            "ppg_start_time": ppg_start_time_text,
            "hz2_first_timestamp": hz2_info["hz2_first_timestamp"],
            "hz2_last_timestamp": hz2_info["hz2_last_timestamp"],
            "ppg_start_time_seconds": ppg_start_time_seconds,
            "hz2_first_time_seconds": hz2_info["hz2_first_time_seconds"],
            "hz2_last_time_seconds": hz2_info["hz2_last_time_seconds"],
            "abs_time_delta_seconds": abs_delta_seconds,
            "timing_eligible_le_10s": bool(timing_eligible),
            "hz2_row_count": hz2_info["hz2_row_count"],
            "hz2_duration_seconds": hz2_info["hz2_duration_seconds"],
            "hz2_non_null_sample_count": hz2_info["hz2_non_null_sample_count"],
            "provisional_ppg_eligible": bool(provisional_ppg_eligible),
        }
    )

alignment_audit_df = pd.DataFrame(audit_rows).sort_values("encounter_id").reset_index(drop=True)
eligible_df = alignment_audit_df.loc[alignment_audit_df["provisional_ppg_eligible"]].copy()

alignment_audit_df.to_csv(OPENOX_ALIGNMENT_AUDIT_CSV, index=False)
eligible_df.to_csv(OPENOX_PPG_ELIGIBLE_CSV, index=False)

summary_df = pd.DataFrame(
    [
        {"metric": "encounters_with_ppg", "value": int(alignment_audit_df["has_ppg_hea"].sum())},
        {"metric": "encounters_with_2hz", "value": int(alignment_audit_df["has_2hz_csv"].sum())},
        {"metric": "encounters_with_bloodgas_so2", "value": int(alignment_audit_df["has_bloodgas_so2"].sum())},
        {"metric": "encounters_with_ppg_and_2hz", "value": int((alignment_audit_df["has_ppg_hea"] & alignment_audit_df["has_2hz_csv"]).sum())},
        {"metric": "delta_le_5s", "value": int((alignment_audit_df["abs_time_delta_seconds"] <= 5).fillna(False).sum())},
        {"metric": "delta_le_10s", "value": int((alignment_audit_df["abs_time_delta_seconds"] <= 10).fillna(False).sum())},
        {"metric": "delta_gt_10s", "value": int((alignment_audit_df["abs_time_delta_seconds"] > 10).fillna(False).sum())},
        {"metric": "provisional_ppg_eligible", "value": int(len(eligible_df))},
    ]
)

print(f"Wrote: {OPENOX_ALIGNMENT_AUDIT_CSV}")
print(f"Wrote: {OPENOX_PPG_ELIGIBLE_CSV}")
summary_df